In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableBranch
import os

In [ ]:
# 使用小米 MiMo 模型
llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
    api_key=os.environ.get("MIMO_API_KEY"),
    temperature=0
)

In [ ]:
# --- 定义子处理器 ---

def booking_handler(request: str) -> str:
    """预订处理器"""
    print("\n--- 转发给预订处理器 ---")
    return f"预订处理器已处理请求: '{request}'. 结果: 模拟预订操作完成."

def info_handler(request: str) -> str:
    """信息处理器"""
    print("\n--- 转发给信息处理器 ---")
    return f"信息处理器已处理请求: '{request}'. 结果: 模拟信息检索完成."

def unclear_handler(request: str) -> str:
    """默认处理器"""
    print("\n--- 处理不明确请求 ---")
    return f"协调员无法处理请求: '{request}'. 请明确您的需求."


In [ ]:
# --- 定义路由链 ---
coordinator_router_prompt = ChatPromptTemplate.from_messages([
    ("system", """分析用户请求，判断应由哪个专家处理：
     - 如果请求与预订航班或酒店相关，输出 'booker'
     - 对于其他一般信息问题，输出 'info'
     - 如果请求不明确或不属于以上类别，输出 'unclear'
     只输出一个单词: 'booker', 'info', 或 'unclear'."""),
    ("user", "{request}")
])

coordinator_router_chain = coordinator_router_prompt | llm | StrOutputParser()


In [ ]:
# --- 定义分支路由逻辑 ---
branches = {
    "booker": RunnablePassthrough.assign(output=lambda x: booking_handler(x['request']['request'])),
    "info": RunnablePassthrough.assign(output=lambda x: info_handler(x['request']['request'])),
    "unclear": RunnablePassthrough.assign(output=lambda x: unclear_handler(x['request']['request'])),
}

# 创建分支路由器
delegation_branch = RunnableBranch(
    (lambda x: x['decision'].strip() == 'booker', branches["booker"]),
    (lambda x: x['decision'].strip() == 'info', branches["info"]),
    branches["unclear"]  # 默认分支
)

# 组合路由链和分支逻辑
coordinator_agent = {
    "decision": coordinator_router_chain,
    "request": RunnablePassthrough()
} | delegation_branch | (lambda x: x['output'])


In [ ]:
# --- 运行示例 ---
print("--- 预订请求示例 ---")
request_a = "帮我订一张去伦敦的机票"
result_a = coordinator_agent.invoke({"request": request_a})
print(f"最终结果 A: {result_a}")

print("\n--- 信息请求示例 ---")
request_b = "意大利的首都是哪里?"
result_b = coordinator_agent.invoke({"request": request_b})
print(f"最终结果 B: {result_b}")

print("\n--- 不明确请求示例 ---")
request_c = "给我讲讲量子物理"
result_c = coordinator_agent.invoke({"request": request_c})
print(f"最终结果 C: {result_c}")
